In [1]:
import os
import sys
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_data
from src.preprocessing import split_and_scale_data

In [8]:
# 1. Load và xóa duplicate
df = load_data("data/creditcard.csv")

# 3. Split + scale
X_train, X_val, X_test, y_train, y_val, y_test, scaler = split_and_scale_data(
    df,
    test_size=0.20,
    validation_size=0.20,
    random_state=42,
)

Successfully loaded dataset


In [9]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

def show_class_distribution(name, y):
    counts = y.value_counts().sort_index()
    print(f"{name}:")
    print(f"  Normal (0): {counts.get(0, 0):,}")
    print(f"  Fraud  (1): {counts.get(1, 0):,}")
    print(f"  Fraud rate : {y.mean():.4%}")

In [10]:
# Dữ liệu gốc sau split — dùng làm baseline
show_class_distribution("Train gốc", y_train)
show_class_distribution("Validation (không resample)", y_val)
show_class_distribution("Test (không resample)", y_test)

Train gốc:
  Normal (0): 169,951
  Fraud  (1): 284
  Fraud rate : 0.1668%
Validation (không resample):
  Normal (0): 56,651
  Fraud  (1): 94
  Fraud rate : 0.1657%
Test (không resample):
  Normal (0): 56,651
  Fraud  (1): 95
  Fraud rate : 0.1674%


In [11]:
smote = SMOTE(
    sampling_strategy=0.1,  # fraud = 10% số giao dịch normal, tránh tăng lên 50% quá mạnh
    random_state=42,
    k_neighbors=5,
)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

show_class_distribution("Train sau SMOTE", y_train_smote)
print("X_train_smote shape:", X_train_smote.shape)

Train sau SMOTE:
  Normal (0): 169,951
  Fraud  (1): 16,995
  Fraud rate : 9.0909%
X_train_smote shape: (186946, 30)


In [12]:
undersampler = RandomUnderSampler(
    sampling_strategy=0.1,  # fraud = 10% normal
    random_state=42,
)

X_train_under, y_train_under = undersampler.fit_resample(X_train, y_train)

show_class_distribution("Train sau undersampling", y_train_under)
print("X_train_under shape:", X_train_under.shape)

Train sau undersampling:
  Normal (0): 2,840
  Fraud  (1): 284
  Fraud rate : 9.0909%
X_train_under shape: (3124, 30)


In [13]:
# 1. Không resample + class_weight="balanced"
X_model_1, y_model_1 = X_train, y_train

# 2. SMOTE
X_model_2, y_model_2 = X_train_smote, y_train_smote

# 3. Undersampling
X_model_3, y_model_3 = X_train_under, y_train_under